# Population Step-Down: 2023 SAL Estimates

**Tess Vu**

Distributes 2023 ward populations down to 2011 SAL geographies using proportional step-down weights (Luo & Wang 2003). Derived from Jill's `newpred.ipynb` with fixes.

Working CRS: EPSG:32735 (UTM 35S), `Shape_Area` is in m².

- Inputs: `2011_census/ea_sal_kzn_gp.shp`, `2023_census/SA_Wards2020.shp`,
           `2023_census/wards_pop.csv`, `sal_w_ward_dedup/sal_w_ward_dedup.shp`
- Output: `notebooks/data/pop_pred_final.csv` (38,380 rows, 21 columns)
- Invariant: sum(sal2023_est) == sum(ward2023_pop) (pycnophylactic)

Maps and diagnostics in `tess_newpred_visuals.ipynb`.

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())))

import geopandas as gpd
import numpy as np
import pandas as pd
from src.paths import WARDS_2020_SHP, WARDS_POP_CSV, SAL_W_WARD_DEDUP, POP_PRED_FINAL

In [3]:
wards = gpd.read_file(WARDS_2020_SHP)
wards_with_pop = pd.read_csv(WARDS_POP_CSV, thousands=",")

# Last row is a "Total" summary line
wards_with_pop = wards_with_pop.drop(wards_with_pop.index[-1])
wards_with_pop = wards_with_pop.rename(columns={"Ward_Code": "WardID", " Total": "Total"})

wards = wards.merge(wards_with_pop[["WardID", "Total"]], on="WardID", how="left")
wards = wards[wards["Province"].isin(["Gauteng", "KwaZulu-Natal"])].copy()

print(f"Wards (GP + KZN): {len(wards)}")

c:\Users\Tess\miniforge3\envs\south_africa\Lib\site-packages\pyogrio\core.py:38: RuntimeWarning: Could not detect GDAL data files. Set GDAL_DATA environment variable to the correct path.
  _init_gdal_data()


Wards (GP + KZN): 1430


In [4]:
sal_with_ward = gpd.read_file(SAL_W_WARD_DEDUP)
sal_with_ward = sal_with_ward.rename(columns={"census_war": "WardID"})

sal_with_ward = sal_with_ward.merge(wards[["WardID", "Total"]], on="WardID", how="left")
print(f"SALs: {len(sal_with_ward)}")

SALs: 38380


In [5]:
# Column selection order here fixes the output CSV column order — do not reorder.
sal_wards = sal_with_ward[
    ["WardID", "EA_CODE", "sal2011_po", "Total", "EA_GTYPE", "EA_TYPE",
     "F4_class", "num_houses", "Black_Afri", "White", "Coloured",
     "Indian_or", "Other", "AREA", "Shape_Area"]
].copy()

# Assign each SAL to the ward with the largest geometric overlap.
sal_wards = (
    sal_wards.sort_values("AREA", ascending=False)
    .drop_duplicates(subset="EA_CODE", keep="first")
    .reset_index(drop=True)
)

print(f"SAL rows after dedup: {len(sal_wards)}")

SAL rows after dedup: 38380


In [6]:
sal_wards = sal_wards.rename(columns={
    "sal2011_po": "sal2011_pop",
    "Total": "ward2023_pop",
    "F4_class": "econ_status",
    "num_houses": "houses2011",
})

# Shape_Area is m² in EPSG:32735
sal_wards["area_km2"] = sal_wards["Shape_Area"] / 1e6
sal_wards = sal_wards.drop(columns=["AREA", "Shape_Area"])

sal_wards["EA_TYPE"] = sal_wards["EA_TYPE"].str.replace(r"_\*$", "", regex=True)
sal_wards["EA_TYPE"] = sal_wards["EA_TYPE"].str.replace("Smallholdings", "Small holdings")

In [7]:
# ZERO-POPULATION IMPUTATION: pop=0 but houses>0 gets Stats SA 2022 average
# household size (3.3). pop=0 and houses=0 stays zero (truly vacant).
HH_SIZE = 3.3

impute_mask = (sal_wards["sal2011_pop"] == 0) & (sal_wards["houses2011"] > 0)
sal_wards.loc[impute_mask, "sal2011_pop"] = sal_wards.loc[impute_mask, "houses2011"] * HH_SIZE

print(f"Imputed {impute_mask.sum()} SALs with pop = 0 but houses > 0.")
print(f"Remaining zero-pop SALs (truly vacant): {(sal_wards['sal2011_pop'] == 0).sum()}")

# Density columns (EDA only, not used in the step-down weight).
sal_wards["sal_dense"] = sal_wards["sal2011_pop"].astype(float) / sal_wards["area_km2"].astype(float)
sal_wards["log_density"] = np.log1p(sal_wards["sal_dense"])

Imputed 1190 SALs with pop = 0 but houses > 0.
Remaining zero-pop SALs (truly vacant): 1270


In [8]:
sal_wards["ward2023_pop"] = pd.to_numeric(sal_wards["ward2023_pop"], errors="coerce")
sal_wards["sal2011_pop"] = pd.to_numeric(sal_wards["sal2011_pop"], errors="coerce")

dupes = sal_wards[sal_wards.duplicated(subset="EA_CODE", keep=False)]
assert len(dupes) == 0, "Unexpected duplicates after dedup step."
print(f"Duplicate EA_CODEs remaining: {len(dupes)}")

Duplicate EA_CODEs remaining: 0


In [9]:
# PROPORTIONAL STEP-DOWN WEIGHT: each SAL's share of its ward's 2011 population.
# share2011 alone is the weight — density re-weighting would double-count the
# 2011 signal (both derive from sal2011_pop).
ward2011_sum = (
    sal_wards.groupby("WardID", as_index=False)["sal2011_pop"].sum()
    .rename(columns={"sal2011_pop": "ward2011_sum"})
)
sal_wards = sal_wards.merge(ward2011_sum, on="WardID", how="left")
sal_wards["share2011"] = sal_wards["sal2011_pop"] / sal_wards["ward2011_sum"]

# Shares must sum to 1 within each ward (pycnophylactic constraint).
ward_share_sums = sal_wards.groupby("WardID")["share2011"].sum()
print(f"Ward share sums — min: {ward_share_sums.min():.6f}, max: {ward_share_sums.max():.6f}")

sal_wards["dasym_weight"] = sal_wards["share2011"]

Ward share sums — min: 1.000000, max: 1.000000


In [10]:
# ESTIMATE 2023 SAL POPULATION and compound annual growth rate (12 years).
sal_wards["sal2023_est"] = sal_wards["dasym_weight"] * sal_wards["ward2023_pop"]
sal_wards["growth_rate"] = (sal_wards["sal2023_est"] / sal_wards["sal2011_pop"]) ** (1 / 12) - 1

print(f"Total estimated 2023 pop: {sal_wards['sal2023_est'].sum():,.0f}")

Total estimated 2023 pop: 27,523,308


In [11]:
# INVARIANT CHECK: step-down must conserve the ward total.
wards["ward2023_pop"] = pd.to_numeric(wards["Total"], errors="coerce")
diff = sal_wards["sal2023_est"].sum() - wards["ward2023_pop"].sum()

print(f"sum(sal2023_est) : {sal_wards['sal2023_est'].sum():,.0f}")
print(f"sum(ward2023_pop): {wards['ward2023_pop'].sum():,.0f}")
print(f"Difference       : {diff:,.2f}")

sum(sal2023_est) : 27,523,308
sum(ward2023_pop): 27,523,308
Difference       : 0.00


In [13]:
sal_wards["EA_CODE"] = sal_wards["EA_CODE"].astype("Int64")

cols = ["sal2023_est", "sal2011_pop", "ward2023_pop", "ward2011_sum", "growth_rate",
        "dasym_weight", "share2011", "log_density", "sal_dense"]
print(sal_wards[cols].describe().to_markdown(floatfmt=",.3f"))

|       |   sal2023_est |   sal2011_pop |   ward2023_pop |   ward2011_sum |   growth_rate |   dasym_weight |   share2011 |   log_density |   sal_dense |
|:------|--------------:|--------------:|---------------:|---------------:|--------------:|---------------:|------------:|--------------:|------------:|
| count |    38,380.000 |    38,380.000 |     38,380.000 |     38,380.000 |    37,110.000 |     38,380.000 |  38,380.000 |    38,380.000 |  38,380.000 |
| mean  |       717.126 |       644.658 |     27,492.234 |     25,318.056 |         0.002 |          0.037 |       0.037 |         7.410 |   7,108.066 |
| std   |       493.443 |       355.077 |     19,153.577 |     13,646.896 |         0.034 |          0.035 |       0.035 |         2.483 |  12,795.790 |
| min   |         0.000 |         0.000 |      1,444.000 |      2,349.000 |        -0.144 |          0.000 |       0.000 |         0.000 |       0.000 |
| 25%   |       373.695 |       453.000 |     11,209.000 |     10,744.900 |       

In [14]:
sal_wards.to_csv(POP_PRED_FINAL, index=False)
print(f"SAVED: {POP_PRED_FINAL}  ({len(sal_wards)} rows, {len(sal_wards.columns)} cols)")
print(list(sal_wards.columns))

SAVED: C:\Users\Tess\Documents\GitHub\south-africa-pharmacy-access\notebooks\data\pop_pred_final.csv  (38380 rows, 21 cols)
['WardID', 'EA_CODE', 'sal2011_pop', 'ward2023_pop', 'EA_GTYPE', 'EA_TYPE', 'econ_status', 'houses2011', 'Black_Afri', 'White', 'Coloured', 'Indian_or', 'Other', 'area_km2', 'sal_dense', 'log_density', 'ward2011_sum', 'share2011', 'dasym_weight', 'sal2023_est', 'growth_rate']
